# Qwen-2.5-3B: Đối Chứng Baseline vs GOR × (AdamW vs GEO-Muon)
Notebook này chạy thí nghiệm đối chứng 2×2 + Base Untouched trên **Qwen-2.5-3B (3.09B, 36 tầng)** trên tập **Suy luận Logic (`GSM8K`, `ARC-Challenge`) + Lập trình (`Python-Alpaca`, `MBPP`)**, sau đó đánh giá đồng thời trên cả **In-Distribution (200 mẫu)** và **External OOD (240 mẫu hoàn toàn khác nguồn & khác format)**:

0. `Qwen-2.5-3B (Base Untouched)` — Mốc gốc chưa fine-tune (~15 giây)
1. `Qwen-2.5-3B (Baseline AdamW)` — Pre-Norm gốc + `AdamW` (đối chứng tiêu chuẩn)
2. `Qwen-2.5-3B (Baseline GEO-Muon)` — Pre-Norm gốc + `GEO-Muon` (tách riêng công lao của `GOR`)
3. `Qwen-2.5-3B + GOR (AdamW)` — Gắn đủ `72` trạm `GOR` + `AdamW`
4. `Qwen-2.5-3B + GOR (GEO-Muon)` — Gắn đủ `72` trạm `GOR` + `GEO-Muon`

Tất cả biến thể trained dùng **chung đúng 1 tập train (4.000 mẫu)**, **chung thứ tự batch (`seed=42`)**, và được đánh giá chéo trên:
- **In-Distribution (200 mẫu):** `GSM8K test`, `ARC-Challenge test`, `MBPP test`, `Python-Alpaca[15000:]`
- **External Out-Of-Distribution (240 mẫu):** `cais/mmlu` (`formal_logic` + `high_school_mathematics`), `openai/openai_humaneval` (raw code), và `Salesforce/wikitext` (`wikitext-2-raw-v1`, chống quên kiến thức tổng quát).

## 1. Kiểm Tra GPU A100 & Cài Thư Viện

In [ ]:
import os
os.system("nvidia-smi")
os.system("pip install -q transformers accelerate datasets matplotlib torch")
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    print("BF16 Supported:", torch.cuda.is_bf16_supported())

## 2. Đồng Bộ Mã Nguồn `qwen_upcycling` Mới Nhất Từ GitHub

In [ ]:
import os
import sys
if not os.path.exists("Alba-model"):
    os.system("git clone https://github.com/potatips/Alba-model.git")
else:
    os.system("cd Alba-model && git pull origin main")
work_dir = os.path.abspath("Alba-model/qwen_upcycling")
if work_dir not in sys.path:
    sys.path.insert(0, work_dir)
print("Loaded modules from:", work_dir)

## 3. Chuẩn Bị Dữ Liệu Train (4.000) + Eval In-Distribution (200) + Eval OOD Ngoài Phân Phối (240)
- **Train + In-Dist Eval:** `openai/gsm8k` + `allenai/ai2_arc` + `python_code_instructions_18k_alpaca` + `mbpp`.
- **External OOD Eval (hoàn toàn khác nguồn & không gắn template train):** `cais/mmlu (formal_logic + high_school_mathematics)` + `openai/openai_humaneval` + `Salesforce/wikitext (wikitext-2-raw-v1)`.

In [ ]:
from qwen_dataset import build_or_load_logic_code_dataset, build_or_load_ood_eval_dataset

DATASET_CACHE = "qwen_logic_code_dataset.json"
OOD_CACHE = "qwen_ood_eval_dataset.json"
dataset_payload = build_or_load_logic_code_dataset(
    cache_path=DATASET_CACHE,
    n_train_logic=2000,
    n_train_code=2000,
    n_eval_logic=100,
    n_eval_code=100,
    seed=42,
)
ood_payload = build_or_load_ood_eval_dataset(
    cache_path=OOD_CACHE,
    n_ood_logic=80,
    n_ood_code=80,
    n_ood_general=80,
    seed=42,
)
print("In-Dist Dataset Summary:", dataset_payload["meta"])
print("OOD Eval Summary:", ood_payload["meta"])
print("\n--- Sample OOD MMLU ---\n", ood_payload["ood_logic"][0]["text"][:300])
print("\n--- Sample OOD HumanEval ---\n", ood_payload["ood_code"][0]["text"][:300])
print("\n--- Sample OOD WikiText-2 ---\n", ood_payload["ood_general"][0]["text"][:300])

## 4. Chạy Đối Chứng 5 Biến Thể (`Base Untouched` + `Baseline` + `GOR` × `AdamW` / `GEO-Muon`)
- Gắn **GOR (`0-Param`, `k=3.0`)** vào đủ `72` trạm residual (`36` Attention + `36` MLP) cho 2 bản `+ GOR`, giữ nguyên Pre-Norm gốc cho 2 bản `Baseline`.
- Tự động lưu sau từng biến thể vào `qwen3b_gor_vs_baseline_results.json` (tự resume nếu Colab ngắt kết nối).

In [ ]:
from qwen_train_upcycled import GOR_VS_BASELINE_VARIANTS, run_qwen3b_ablation_suite

results = run_qwen3b_ablation_suite(
    model_name="Qwen/Qwen2.5-3B",
    total_steps=200,
    seq_len=512,
    micro_batch_size=2,
    grad_accum_steps=8,
    n_train_logic=2000,
    n_train_code=2000,
    n_eval_logic=100,
    n_eval_code=100,
    adamw_lr=5e-5,
    muon_lr=1e-4,
    dataset_cache_path="qwen_logic_code_dataset.json",
    ood_cache_path="qwen_ood_eval_dataset.json",
    n_ood_per_domain=80,
    eval_ood=True,
    results_cache_path="qwen3b_gor_vs_baseline_results.json",
    output_html_path="qwen3b_gor_vs_baseline_report.html",
    output_png_path="qwen3b_gor_vs_baseline_grid.png",
    save_checkpoint_dir=None,
    variants=GOR_VS_BASELINE_VARIANTS,
    seed=42,
)

## 5. Xem Biểu Đồ & Tải Về Báo Cáo So Sánh (`qwen3b_gor_vs_baseline_report.html`)

In [ ]:
from IPython.display import Image, display
if os.path.exists("qwen3b_gor_vs_baseline_grid.png"):
    display(Image("qwen3b_gor_vs_baseline_grid.png"))
try:
    from google.colab import files
    files.download("qwen3b_gor_vs_baseline_report.html")
    files.download("qwen3b_gor_vs_baseline_results.json")
    files.download("qwen3b_gor_vs_baseline_grid.png")
except Exception:
    print("Report saved locally at qwen3b_gor_vs_baseline_report.html")